# CanvasXpress in VS Code — Python

This notebook uses the **native** `canvasxpress` Python API — you build a `CanvasXpress`
object and render it with `CXNoteBook`, exactly as you would in Jupyter or JupyterLab. There is
no helper function and no hand-written mime bundle.

The **CanvasXpress Notebook Renderer** extension reads the package's own output, recovers the
`{data, config}` spec from it, and draws a live interactive chart in the cell.

> **Note on `display(*...)`:** `CXNoteBook(...).render()` *returns* a list of display objects
> rather than displaying them, so on its own it prints `[<HTML object>, ...]` and nothing is
> drawn — in any Jupyter frontend, not just VS Code. Unpacking it into `display()` is what
> hands those objects to the notebook.
>
> **Requirements:** `pip install canvasxpress pandas` and the CanvasXpress Notebook Renderer
> extension. Select a Python kernel before running.
>
> Each chart produces a canvas output and a script output; the renderer draws the chart from
> the latter. If you see raw markup instead of a chart, click the **···** on the output and
> choose **Change Presentation → CanvasXpress**.

## 1. Bar chart

In [ ]:
from canvasxpress.canvas import CanvasXpress
from canvasxpress.render.jupyter import CXNoteBook
from IPython.display import display

chart = CanvasXpress(
    render_to="bar1",
    data={"y": {
        "vars": ["Revenue"],
        "smps": ["Q1", "Q2", "Q3", "Q4"],
        "data": [[10, 14, 9, 17]],
    }},
    config={
        "graphType": "Bar",
        "title": "Quarterly Revenue",
        "xAxis": ["Revenue"],
        "smpTitle": "Quarter",
        "showLegend": False,
    },
)

display(*CXNoteBook(chart).render())

## 2. Scatter plot

For `Scatter2D` the `smps` are the axis dimensions and the `vars` are the point ids — one row
of data per point, its columns matching `smps`.

In [ ]:
chart = CanvasXpress(
    render_to="scatter1",
    data={"y": {
        "smps": ["Weight", "MPG"],
        "vars": ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8"],
        "data": [
            [3.2, 24], [2.8, 30], [4.1, 18], [3.6, 21],
            [2.4, 33], [3.9, 19], [2.1, 36], [4.5, 15],
        ],
    }},
    config={
        "graphType": "Scatter2D",
        "title": "Weight vs. MPG",
        "xAxis": ["Weight"],
        "yAxis": ["MPG"],
    },
)

display(*CXNoteBook(chart).render())

## 3. Straight from a `DataFrame`

Usually the data is already a pandas `DataFrame` rather than a hand-written `y` object. Pass it
to `CanvasXpress` directly — the package converts it for you: **the index becomes `vars`, the
columns become `smps`**, and the values become the data matrix. That is the layout a Heatmap
(and most CanvasXpress graph types) expects — features in rows, samples in columns.

In [ ]:
import pandas as pd

expr = pd.DataFrame(
    {
        "S1": [1.2, -0.9, 0.5, 1.7],
        "S2": [-0.4, 1.4, 0.6, -1.2],
        "S3": [0.8, 0.2, -1.3, 0.9],
        "S4": [1.9, -0.7, 1.1, -0.6],
        "S5": [-1.1, 1.8, -0.2, 0.4],
        "S6": [0.3, -0.5, 1.6, -1.4],
    },
    index=["Gene A", "Gene B", "Gene C", "Gene D"],
)

chart = CanvasXpress(
    render_to="heat1",
    data=expr,
    config={
        "graphType": "Heatmap",
        "title": "Expression Heatmap",
        "samplesClustered": True,
        "variablesClustered": True,
    },
)

display(*CXNoteBook(chart).render())

## 4. Several charts in one output

`CXNoteBook` accepts more than one chart. Each needs its own `render_to` target; the renderer
draws them all in the same cell output.

In [ ]:
left = CanvasXpress(
    render_to="multi_a",
    data={"y": {"vars": ["Revenue"], "smps": ["Q1", "Q2", "Q3", "Q4"], "data": [[10, 14, 9, 17]]}},
    config={"graphType": "Bar", "title": "Revenue", "showLegend": False},
)

right = CanvasXpress(
    render_to="multi_b",
    data={"y": {"vars": ["Cost"], "smps": ["Q1", "Q2", "Q3", "Q4"], "data": [[7, 8, 8, 11]]}},
    config={"graphType": "Line", "title": "Cost", "showLegend": False},
)

display(*CXNoteBook(left, right).render(columns=2))